# Stickler + Strands Evals on a real extraction dataset

Strands Evals scores structured output with `Equals` — whole-object equality, 0.0 or 1.0. On real
documents almost nothing matches the labels exactly, so `Equals` collapses to a near-constant near-zero
and cannot rank extractors or say which field is broken. Stickler scores the same outputs field by
field, so it does both.

53 FCC political-advertising invoices, extracted by Claude Haiku 4.5 from OCR text, scored by `Equals`
and by stickler on identical outputs. Requires AWS credentials with Bedrock access.

- **Setup** — imports, then one data-processing cell (dataset -> model -> extraction -> cases)
- **The payoff** — `Equals` vs `StructuredOutputEvaluator`
- **Which field is broken** — per-field rollup from the evaluator

## Setup

`stickler.eval_for(Model)` compiles a field-level comparison plan from a Pydantic model.
`StructuredOutputEvaluator` wraps it as a Strands Evals `Evaluator`, imported from
[../integrations/strands_evals/](../integrations/strands_evals/) so the demo and the proposed upstream
code stay in sync.

```
uv run --with strands-agents-evals jupyter lab
```

In [1]:
import importlib.metadata as md
import json
import sys
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import List, Optional

import boto3
from pydantic import BaseModel, Field
from strands import Agent
from strands.models import BedrockModel

import stickler
from strands_evals import Case, Experiment
from strands_evals.evaluators import Equals

sys.path.insert(0, str((Path.cwd() / ".." / "integrations" / "strands_evals").resolve()))
from stickler_evaluator import StructuredOutputEvaluator  # noqa: E402

for dist in ("stickler-eval", "strands-agents-evals", "strands-agents"):
    print(f"{dist:<21} {md.version(dist)}")

stickler-eval         0.5.0
strands-agents-evals  1.0.3
strands-agents        1.50.2


## Data: dataset -> model -> extraction -> cases

One cell, in the order the pipeline runs:

1. **Dataset** — 53 invoices from [RealKIE-FCC-Verified](https://huggingface.co/datasets/amazon-agi/RealKIE-FCC-Verified), fetched over HTTP (documents with >25 line items are skipped; Hungarian matching is cubic in list length).
2. **Model** — a plain `FCCInvoice`; every nullable field is `Optional` because on scanned invoices those fields are legitimately absent, and the evaluator must score "correctly returned nothing" as a success.
3. **Extraction** — a Strands agent reads OCR text and fills `FCCInvoice`, 8 documents at a time. Live billable Bedrock calls on every run.
4. **Cases** — one per document; `task` replays the stored extraction so both evaluators score identical outputs.

In [2]:
# --- Dataset ---------------------------------------------------------------
DATASET = "amazon-agi/RealKIE-FCC-Verified"
MAX_LINE_ITEMS = 25


def fetch_rows() -> List[dict]:
    base = (
        "https://datasets-server.huggingface.co/rows"
        f"?dataset={urllib.parse.quote(DATASET, safe='')}&config=default&split=test"
    )
    rows, offset = [], 0
    while True:
        with urllib.request.urlopen(f"{base}&offset={offset}&length=25", timeout=60) as resp:
            payload = json.load(resp)
        batch = payload.get("rows", [])
        rows.extend(item["row"] for item in batch)
        total = payload.get("num_rows_total")
        offset += 25
        if not batch or (total is not None and len(rows) >= total):
            return rows


all_rows = fetch_rows()
scored_rows = [r for r in all_rows if len(r["json_response"].get("LineItems") or []) <= MAX_LINE_ITEMS]


# --- Model -----------------------------------------------------------------
class FCCLineItem(BaseModel):
    LineItemDescription: Optional[str] = None
    LineItemStartDate: Optional[str] = None
    LineItemEndDate: Optional[str] = None
    LineItemDays: Optional[str] = None
    LineItemRate: Optional[float] = None


class FCCInvoice(BaseModel):
    Agency: str
    Advertiser: str
    GrossTotal: Optional[float] = None
    PaymentTerms: Optional[str] = None
    AgencyCommission: Optional[float] = None
    NetAmountDue: Optional[float] = None
    LineItems: List[FCCLineItem] = Field(default_factory=list)


GROUND_TRUTH = {r["id"]: FCCInvoice.model_validate(r["json_response"]) for r in scored_rows}


# --- Extraction (live Bedrock) ---------------------------------------------
MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
OCR_CHAR_LIMIT = 12_000
MAX_WORKERS = 8

# Uses the profile the kernel was launched with; relaunch with AWS_PROFILE=<profile> if unset.
SESSION = boto3.Session(region_name="us-east-1")
SESSION.client("sts").get_caller_identity()   # free, fails fast if the token is stale
BEDROCK = BedrockModel(model_id=MODEL, boto_session=SESSION)

EXTRACT_PROMPT = (
    "Extract the invoice fields from this FCC political advertising invoice. Return null for any "
    "field not shown on the document. Transcribe dates exactly as printed. Include every row of the "
    "line-item table.\n\nDOCUMENT:\n{text}"
)


def extract(row: dict) -> FCCInvoice:
    agent = Agent(model=BEDROCK, system_prompt="You extract invoice data.", callback_handler=None)
    result = agent(EXTRACT_PROMPT.format(text=row["text"][:OCR_CHAR_LIMIT]), structured_output_model=FCCInvoice)
    if result.structured_output is None:
        raise ValueError("agent returned no structured output")
    return result.structured_output


print(f"extracting {len(scored_rows)} documents via {MODEL.split('.')[-1]} (billable)...")

PREDICTIONS = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(extract, row): row for row in scored_rows}
    for done, future in enumerate(as_completed(futures), 1):
        PREDICTIONS[futures[future]["id"]] = future.result()
        if done % 10 == 0 or done == len(scored_rows):
            print(f"    {done}/{len(scored_rows)}")


# --- Cases -----------------------------------------------------------------
cases = [
    Case(
        name=row["id"][:12],
        input=row["text"][:OCR_CHAR_LIMIT],
        expected_output=GROUND_TRUTH[row["id"]],
        metadata={"doc_id": row["id"]},
    )
    for row in scored_rows
]


def task(case: Case) -> FCCInvoice:
    return PREDICTIONS[case.metadata["doc_id"]]


print(f"ready: {len(cases)} cases")

extracting 53 documents via claude-haiku-4-5-20251001-v1:0 (billable)...


    10/53


    20/53


    30/53


    40/53


    50/53


    53/53
ready: 53 cases


## The payoff: `Equals` vs `StructuredOutputEvaluator`

The demo. Same cases, same harness, same extractions — only the evaluator differs. `Equals` is the
strongest deterministic evaluator Strands Evals ships for structured output.

`run_evaluations` wraps `asyncio.run`, which raises inside a Jupyter kernel, so the async variant is
used here.

In [3]:
stickler_eval = StructuredOutputEvaluator(FCCInvoice)

stickler_report = await Experiment(cases=cases, evaluators=[stickler_eval]).run_evaluations_async(task, max_workers=1)
equals_report = await Experiment(cases=cases, evaluators=[Equals()]).run_evaluations_async(task, max_workers=1)

stickler_scores = dict(zip((c["name"] for c in stickler_report.cases), stickler_report.scores))
equals_scores = dict(zip((c["name"] for c in equals_report.cases), equals_report.scores))
reasons = dict(zip((c["name"] for c in stickler_report.cases), stickler_report.reasons))

print(f"{len(cases)} documents")
print(f"  stickler overall {stickler_report.overall_score:.3f}")
print(f"  equals   overall {equals_report.overall_score:.3f}\n")

SHOW = 15
ranked = sorted(stickler_scores, key=stickler_scores.get)
print(f"{'document':14} {'stickler':>9} {'equals':>7}   weakest fields")
print("-" * 78)
for name in ranked[:SHOW]:
    print(f"{name:14} {stickler_scores[name]:>9.3f} {equals_scores[name]:>7.1f}   {reasons[name][:38]}")
print(f"... {len(ranked) - SHOW} more\n")

s_distinct = len({round(v, 4) for v in stickler_scores.values()})
e_distinct = len({round(v, 4) for v in equals_scores.values()})
print(f"distinct scores   stickler {s_distinct:>3}   equals {e_distinct:>3}")
print("Equals resolves the corpus into one or two values; stickler spreads it across the range, so only")
print("stickler can rank extractors, detect a regression, or point at a field.")

53 documents
  stickler overall 0.783
  equals   overall 0.000

document        stickler  equals   weakest fields
------------------------------------------------------------------------------
3aac2d4d386a       0.371     0.0   weakest fields: Agency=0.00; Advertise
ad34d418249c       0.486     0.0   weakest fields: GrossTotal=0.00; Agenc
f3eb65041a1a       0.504     0.0   weakest fields: GrossTotal=0.00; Agenc
75986ca8ed4b       0.506     0.0   weakest fields: Agency=0.00; Advertise
2de731d64adf       0.514     0.0   weakest fields: PaymentTerms=0.00; Age
caa7473ca263       0.543     0.0   weakest fields: Advertiser=0.00; Payme
7ec895784100       0.571     0.0   weakest fields: Agency=0.00; Advertise
c86d3402da03       0.599     0.0   weakest fields: GrossTotal=0.00; NetAm
cfa6bc56d5a2       0.608     0.0   weakest fields: Agency=0.00; PaymentTe
21762bcc6102       0.623     0.0   weakest fields: PaymentTerms=0.00; Age
b55d327a4141       0.634     0.0   weakest fields: Agency=0.00; Net

## Which field is broken

A document score says how bad the extractor is; a per-field rollup says what to fix. `EvaluationOutput`
carries only four scalar fields, so this rollup cannot cross the Strands harness boundary — the
evaluator exposes it directly via `aggregate()`, keeping the logic in the class instead of an inline
loop in the notebook.

In [4]:
rollup = stickler_eval.aggregate((GROUND_TRUTH[c.metadata["doc_id"]], PREDICTIONS[c.metadata["doc_id"]]) for c in cases)

print(f"{'field':<20} {'mean':>6} {'worst':>6} {'perfect':>9} {'below thr':>10}  comparator")
print("-" * 82)
for field, m in rollup.items():
    print(f"{field:<20} {m['mean']:>6.3f} {m['worst']:>6.3f} "
          f"{m['perfect']:>4}/{m['count']:<4} {m['below_threshold']:>10}  "
          f"{m['comparator'].replace('Comparator', '')}")

worst = next(iter(rollup))   # aggregate() returns worst mean first
print(f"\nWorst field: {worst} ({rollup[worst]['mean']:.3f}) -- the effort belongs here, "
      f"not on fields already scoring high.")

field                  mean  worst   perfect  below thr  comparator
----------------------------------------------------------------------------------
PaymentTerms          0.642  0.000   34/53           19  Levenshtein
LineItems             0.681  0.000    5/53           25  Hungarian (per-element StructuredModel)
Agency                0.746  0.000   36/53           13  Levenshtein
NetAmountDue          0.792  0.000   42/53           11  Numeric
Advertiser            0.828  0.000   30/53            7  Levenshtein
AgencyCommission      0.849  0.000   45/53            8  Numeric
GrossTotal            0.943  0.000   50/53            3  Numeric

Worst field: PaymentTerms (0.642) -- the effort belongs here, not on fields already scoring high.
